# Vector DataBase: ChromaDB
To setup the vector databse for the `RAG System`, this notebook should be use to setup the Vector Databse using `ChromaDB`. At the end of the notebook is an example of the retrival information from the batabase

In [1]:
#pip install pathlib
#%pip install langchain_core
#%pip install langchain_ollama
#%pip install langchain_text_splitters
#%pip install langchain_chroma

## Import Librarys

In [2]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

## From Markdown to Document
This cell is just an example of how to convert the `.md` with the different BMW models data to `Langchain Documents`

In [3]:
#from pathlib import Path
#from langchain_core.documents import Document


# Find the root from the run folder or its parent directories.
project_path = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "Documents").is_dir() and (folder / "Notebooks").is_dir()
)

path = project_path / "Documents" / "BMW_i3.md"

texto = path.read_text(encoding="utf-8")

documento = Document(
    page_content=texto,
    metadata={"source": str(path)}
)

print("File:", documento.metadata["source"])
print("Characters lenght:", len(documento.page_content))
print(documento.page_content[:1000])

File: /Users/andreyvargassolis/Desktop/Data_Science/CarSeller_AI_RAG_System/Documents/BMW_i3.md
Characters lenght: 5095
---
model: "BMW i3"
manufacturer: "BMW"
market: "United States"
language: "en"
review_date: "2026-10-31"
source_domain: "bmwusa.com"
---

# BMW i3

BMW USA model and technical information dossier

## Scope

This document consolidates model variants, official published specifications, highlighted features and references from BMW's United States website. It is a factual summary, not a verbatim reproduction of the manufacturer's promotional copy. The live source pages remain authoritative for interactive configurations, photographs, offers and legal terms.

## Key published facts

- For the 2027 i3 50 xDrive, BMW publishes an estimated 446–468-mile range, up to 463 hp and 0–60 mph in 4.5 seconds.
- BMW also lists maximum DC fast-charging power of 400 kW, an 800-volt architecture and NACS charging hardware.

## Published variants and prices

The reviewed model page does n

## Create the Vector Database & Embeddings settings
This cell search if the vector database is not created, in that case the DB would be establish in the project with `ChormaDB `and would use the model `embeddinggemma` to assign the embeddigns to the fragments of the documents

In [4]:
#from pathlib import Path
#from langchain_core.documents import Document
#from langchain_ollama import OllamaEmbeddings
#from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="embeddinggemma")

vector_db = Chroma(
    collection_name="BMW_Models",
    embedding_function=embeddings,
    persist_directory=str(project_path / "chroma_db")
)


## Convert .md into Documents & Fragment


In [5]:
#from pathlib import Path
#from langchain_core.documents import Document
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_chroma import Chroma

path_md_models = project_path / "Documents"

documents = []

for markdown in path_md_models.glob("*.md"):
    text = markdown.read_text(encoding="utf-8")

    document = Document(
        page_content=text,
        metadata={
            "source": str(markdown),
            "model": markdown.stem
        }
    )

    documents.append(document)

if not documents:
    raise ValueError("No documents found in the specified directory.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

print("Original Docs:", len(documents))
print("Generated Fragments:", len(chunks))
print("\nFirst Fragment:")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)



Original Docs: 41
Generated Fragments: 381

First Fragment:
---
model: "BMW X7 M60i"
manufacturer: "BMW"
market: "United States"
language: "en"
review_date: "2026-10-31"
source_domain: "bmwusa.com"
---

# BMW X7 M60i

BMW USA model and technical information dossier

## Scope

This document consolidates model variants, official published specifications, highlighted features and references from BMW's United States website. It is a factual summary, not a verbatim reproduction of the manufacturer's promotional copy. The live source pages remain authoritative for interactive configurations, photographs, offers and legal terms.

## Published variants and prices

| Variant | 0–60 mph | Horsepower | Starting US MSRP |
|---|---:|---:|---:|
| X7 M60i | 4.5 sec | 523 | $115,000 |

## Technical specifications by variant

### X7 M60i

**Manufacturer model code:** 27SL

Metadata: {'source': '/Users/andreyvargassolis/Desktop/Data_Science/CarSeller_AI_RAG_System/Documents/BMW_X7_M60i.md', 'model': 'BM

`split_documents` returns objects that are `Document` like the original ones, they conserve the original metadata too.

## Generate Embeddings & Insert them on the Vector Database

In [6]:
# Reset the collection before adding new documents
vector_db.reset_collection()

ids = vector_db.add_documents(chunks)

print("Documents added to the vector database:", len(ids))

Documents added to the vector database: 381


## Search with Retrieval Information from Database
This example use the model `embeddinggemma` to search similarity with the chunks stored in the database

In [7]:
results = vector_db.similarity_search(
    "What is the range of the BMW i3?",
    k=2
)

for doc in results:
    print("Metadata:", doc.metadata["source"])
    print(doc.page_content)
    print("\n------------------------------\n")

Metadata: /Users/andreyvargassolis/Desktop/Data_Science/CarSeller_AI_RAG_System/Documents/BMW_i3.md
---
model: "BMW i3"
manufacturer: "BMW"
market: "United States"
language: "en"
review_date: "2026-10-31"
source_domain: "bmwusa.com"
---

# BMW i3

BMW USA model and technical information dossier

## Scope

This document consolidates model variants, official published specifications, highlighted features and references from BMW's United States website. It is a factual summary, not a verbatim reproduction of the manufacturer's promotional copy. The live source pages remain authoritative for interactive configurations, photographs, offers and legal terms.

## Key published facts

- For the 2027 i3 50 xDrive, BMW publishes an estimated 446–468-mile range, up to 463 hp and 0–60 mph in 4.5 seconds.
- BMW also lists maximum DC fast-charging power of 400 kW, an 800-volt architecture and NACS charging hardware.

## Published variants and prices

The reviewed model page does not provide a structu